# MSPCIFormer — Optuna Hyperparameter Tuning

Runs hyperparameter search for all models using Optuna on Google Colab.

**Models:** MSPCIFormer, TimeXer, MSGNet, TimesNet, NBeats (interpretable + generic), iTransformer, DLinear

**Steps:**
1. Mount Google Drive and clone / pull the repo
2. Install dependencies
3. Configure tuning settings
4. Run tuning per model
5. Analyse results


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone repo and set working directory

Option A: clone fresh from GitHub (first time)  
Option B: pull latest if already cloned to Drive


In [ ]:
import os

REPO_URL   = 'https://github.com/cmajorsolo/MSPCIFormer.git'
BRANCH     = 'nbeats_timesnet'
DRIVE_PATH = '/content/drive/MyDrive/MSPCIFormer'

if os.path.exists(DRIVE_PATH):
    print('Repo found on Drive - pulling latest...')
    %cd {DRIVE_PATH}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    print('Cloning repo to Drive...')
    !git clone --branch {BRANCH} {REPO_URL} {DRIVE_PATH}
    %cd {DRIVE_PATH}

print('Working directory:', os.getcwd())


In [ ]:
# Verify data file exists before running tuning
import os
data_file_path = os.path.join(ROOT_PATH if 'ROOT_PATH' in dir() else './data/', 
                               DATA_FILE if 'DATA_FILE' in dir() else 'crypto_prices_wide.csv')
# Use the configured values if already set
root = './data/'
fname = 'crypto_prices_wide.csv'
full_path = os.path.join(root, fname)
if os.path.exists(full_path):
    print(f'Data file found: {full_path}')
else:
    raise FileNotFoundError(
        f'Data file not found: {full_path}\n'
        f'Please upload crypto_prices_wide.csv to {root} before tuning.'
    )


## 3. Install dependencies

In [ ]:
!pip install optuna scikit-learn --quiet
print('Done.')


## 4. Verify GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))


## 5. Tuning configuration

Edit these settings before running the tuning cells.


In [ ]:
# ── Tuning settings ──────────────────────────────────────
N_TRIALS     = 50   # Optuna trials per model
TRAIN_EPOCHS = 5    # epochs per trial
DATA_FILE    = 'crypto_prices_wide.csv'
ROOT_PATH    = './data/'
# ─────────────────────────────────────────────────────────

print(f'Trials per model : {N_TRIALS}')
print(f'Epochs per trial : {TRAIN_EPOCHS}')
print(f'Data file        : {DATA_FILE}')


## 6. Helper — run tuning for one model

In [ ]:
import subprocess, sys

def tune_model(model, nbeats_type=None, n_trials=None, train_epochs=None):
    t = n_trials or N_TRIALS
    e = train_epochs or TRAIN_EPOCHS
    cmd = (
        f'python tuning/tune_hyperparams.py'
        f' --model {model}'
        f' --n_trials {t}'
        f' --train_epochs {e}'
        f' --data_path {DATA_FILE}'
        f' --root_path {ROOT_PATH}'
    )
    if nbeats_type:
        cmd += f' --nbeats_type {nbeats_type}'
    label = model + (f' ({nbeats_type})' if nbeats_type else '')
    print('\n' + '='*60 + f'\nTuning: {label}\nCommand: {cmd}\n' + '='*60)
    sys.stdout.flush()
    # Stream output line by line so it appears in real time in Colab
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'tune_model({label}) failed with return code {proc.returncode}')


## 7. Run tuning — one cell per model

Each cell is independent. Run individually or use **Run All**.  
Results are saved to `tuning/results/` after each model completes.


In [ ]:
# DLinear
tune_model('DLinear')


In [ ]:
# NBeats - interpretable
tune_model('NBeats', nbeats_type='interpretable')


In [ ]:
# NBeats - generic
tune_model('NBeats', nbeats_type='generic')


In [ ]:
# TimesNet
tune_model('TimesNet')


In [ ]:
# MSGNet
tune_model('MSGNet')


In [ ]:
# iTransformer
tune_model('iTransformer')


In [ ]:
# TimeXer
tune_model('TimeXer')


In [ ]:
# MSPCIFormer
tune_model('MSPCIFormer')


## 8. View all best params

In [ ]:
import json, glob

result_files = sorted(glob.glob('./tuning/results/*_best_params.json'))
if not result_files:
    print('No results found yet. Run the tuning cells above first.')
else:
    for path in result_files:
        with open(path) as f:
            data = json.load(f)
        print('\n' + '='*50)
        print('Model     :', data['model_key'])
        print('Best loss :', round(data['best_vali_loss'], 7))
        print('Params    :')
        for k, v in data['best_params'].items():
            print(f'  {k}: {v}')


## 9. Analyse a specific study

Change `MODEL` to whichever model you want to inspect.


In [ ]:
MODEL = 'MSPCIFormer'  # options: MSPCIFormer, TimeXer, MSGNet, TimesNet, iTransformer, DLinear
                       # for NBeats: pass --model NBeats --nbeats_type interpretable
os.system(f'python tuning/analyze_study.py --model {MODEL}')


## 10. Copy results to Google Drive

In [ ]:
# Run only if repo was cloned to /content (not Drive) to avoid losing results
import shutil
SAVE_TO = '/content/drive/MyDrive/MSPCIFormer_tuning_results'
os.makedirs(SAVE_TO, exist_ok=True)
shutil.copytree('./tuning/results', SAVE_TO, dirs_exist_ok=True)
print('Results copied to', SAVE_TO)
